# EDA — Boxplots cho ROA, Vòng quay tài sản, Dòng tiền và Khả năng thanh toán

Ô này đọc dữ liệu và hiển thị các cột tìm được, ô tiếp theo vẽ boxplot. Nếu có lỗi, copy traceback cho tôi.

In [30]:
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(style="whitegrid")

# Tìm file CSV (hỗ trợ chạy từ workspace root hoặc từ thư mục notebooks)
candidates = [Path("data/dulieuketoan_clean.csv"), Path("..") / "data" / "dulieuketoan_clean.csv"]
csv_path = None
for p in candidates:
    if p.exists():
        csv_path = p
        break
if csv_path is None:
    raise FileNotFoundError("Không tìm thấy file 'dulieuketoan_clean.csv' trong 'data/' hoặc '../data/'")

# Đọc dữ liệu
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

print('Đã đọc:', csv_path)
print('Tổng số cột =', len(df.columns))
print('Các cột (ví dụ 40 đầu):')
print(df.columns.tolist()[:40])

Đã đọc: ..\data\dulieuketoan_clean.csv
Tổng số cột = 96
Các cột (ví dụ 40 đầu):
['ROA(C) before interest and depreciation before interest', 'ROA(A) before interest and % after tax', 'ROA(B) before interest and depreciation after tax', 'Operating Gross Margin', 'Realized Sales Gross Margin', 'Operating Profit Rate', 'Pre-tax net Interest Rate', 'After-tax net Interest Rate', 'Non-industry income and expenditure/revenue', 'Continuous interest rate (after tax)', 'Operating Expense Rate', 'Research and development expense rate', 'Cash flow rate', 'Interest-bearing debt interest rate', 'Tax rate (A)', 'Net Value Per Share (B)', 'Net Value Per Share (A)', 'Net Value Per Share (C)', 'Persistent EPS in the Last Four Seasons', 'Cash Flow Per Share', 'Revenue Per Share (Yuan ¥)', 'Operating Profit Per Share (Yuan ¥)', 'Per Share Net profit before tax (Yuan ¥)', 'Realized Sales Gross Profit Growth Rate', 'Operating Profit Growth Rate', 'After-tax Net Profit Growth Rate', 'Regular Net Profit Growt

In [31]:
# Các tên cột mong muốn (theo header trong file)
wanted = {
    'roa': [
        'ROA(C) before interest and depreciation before interest',
        'ROA(A) before interest and % after tax',
        'ROA(B) before interest and depreciation after tax'
    ],
    'turnover': ['Total Asset Turnover'],
    'cash': ['Cash flow rate'],
    'liquidity': ['Current Ratio', 'Quick Ratio']
}

def existing(cols):
    return [c for c in cols if c in df.columns]

roa_cols = existing(wanted['roa'])
turnover_cols = existing(wanted['turnover'])
cash_cols = existing(wanted['cash'])
liquidity_cols = existing(wanted['liquidity'])

print('\nTìm được cột:')
print('ROA:', roa_cols)
print('Vòng quay tài sản:', turnover_cols)
print('Dòng tiền:', cash_cols)
print('Khả năng thanh toán:', liquidity_cols)

# Chuyển sang kiểu số (nếu cần)
for col in (roa_cols + turnover_cols + cash_cols + liquidity_cols):
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Vẽ 2x2 boxplots
fig, axes = plt.subplots(2,2, figsize=(14,10))
axes = axes.flatten()
groups = [
    ("ROA (các biến)", roa_cols),
    ("Vòng quay tài sản", turnover_cols),
    ("Dòng tiền từ HĐKD", cash_cols),
    ("Khả năng thanh toán", liquidity_cols),
]

for ax, (title, cols) in zip(axes, groups):
    if not cols:
        ax.set_title(f"{title} (no columns found)")
        ax.axis('off')
        continue
    if len(cols) == 1:
        sns.boxplot(x=df[cols[0]].dropna(), ax=ax)
        ax.set_xlabel(cols[0])
    else:
        data = df[cols].melt(value_name='value', var_name='indicator')
        sns.boxplot(x='indicator', y='value', data=data, ax=ax)
        labels = list(data['indicator'].unique())
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_title(title)

plt.tight_layout()
# Lưu ảnh vào cùng thư mục notebooks để dễ tìm
out_path = Path('02_boxplots.png')
# đảm bảo folder tồn tại (nếu chạy từ root project)
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, dpi=150)
print(f'Đã lưu hình: {out_path.resolve()}')
plt.show()



Tìm được cột:
ROA: ['ROA(C) before interest and depreciation before interest', 'ROA(A) before interest and % after tax', 'ROA(B) before interest and depreciation after tax']
Vòng quay tài sản: ['Total Asset Turnover']
Dòng tiền: ['Cash flow rate']
Khả năng thanh toán: ['Current Ratio', 'Quick Ratio']
Đã lưu hình: C:\Users\Admin\Financial-Distress-Prediction\notebooks\02_boxplots.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_24224\3149357841.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [32]:
# Phân tích chi tiết và vẽ biểu đồ nâng cao cho từng nhóm
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
sns.set(style='whitegrid')

# Load dữ liệu (độc lập ô này để dễ chạy riêng)
csv = Path('data') / 'dulieuketoan_clean.csv'
if not csv.exists():
    csv = Path('..') / 'data' / 'dulieuketoan_clean.csv'
    if not csv.exists():
        raise FileNotFoundError('Không tìm thấy data/dulieuketoan_clean.csv')

df = pd.read_csv(csv)
df.columns = df.columns.str.strip()

groups = {
    'ROA': [
        'ROA(C) before interest and depreciation before interest',
        'ROA(A) before interest and % after tax',
        'ROA(B) before interest and depreciation after tax'
    ],
    'Vòng quay tài sản': ['Total Asset Turnover'],
    'Dòng tiền từ HĐKD': ['Cash flow rate'],
    'Khả năng thanh toán': ['Current Ratio', 'Quick Ratio']
}

# Giúp chọn các cột tồn tại
for k, cols in groups.items():
    groups[k] = [c for c in cols if c in df.columns]

# Hàm loại outlier theo IQR
def remove_outliers(series, k=1.5):
    series = series.dropna()
    if series.empty:
        return series
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    low = q1 - k * iqr
    high = q3 + k * iqr
    return series[(series >= low) & (series <= high)]

# Hàm vẽ chi tiết cho một nhóm
def plot_detailed(name, cols, save=True):
    if not cols:
        print(f'No columns for {name}')
        return
    # chuẩn hoá dữ liệu số
    data = df[cols].apply(pd.to_numeric, errors='coerce')
    desc = data.describe().T
    desc['skew'] = data.skew()
    desc['kurt'] = data.kurt()
    desc['missing_pct'] = data.isna().mean() * 100

    fig = plt.figure(constrained_layout=True, figsize=(14,10))
    gs = fig.add_gridspec(3, 3)
    ax0 = fig.add_subplot(gs[0, :])   # violin + box
    ax1 = fig.add_subplot(gs[1, :2])  # histogram + kde
    ax2 = fig.add_subplot(gs[1, 2])   # box without outliers
    ax3 = fig.add_subplot(gs[2, :])   # stats table

    # Violin + box overlay
    sns.violinplot(data=data, inner=None, color='0.9', ax=ax0)
    sns.boxplot(data=data, width=0.15, fliersize=3, ax=ax0)
    ax0.set_title(f'{name} — Violin + Box')
    ax0.tick_params(axis='x', rotation=30)

    # Histogram + KDE per column
    for col in cols:
        col_series = pd.to_numeric(df[col], errors='coerce').dropna()
        if col_series.empty:
            continue
        sns.histplot(col_series, kde=True, stat='density', label=col, ax=ax1, alpha=0.5, bins=30)
    ax1.set_title('Histogram + KDE')
    ax1.legend(fontsize='small')

    # Boxplot sau khi loại outliers (IQR)
    clean = {c: remove_outliers(pd.to_numeric(df[c], errors='coerce')) for c in cols}
    clean_df = pd.DataFrame(clean)
    sns.boxplot(data=clean_df, ax=ax2)
    ax2.set_title('Boxplot (IQR trimmed)')
    ax2.tick_params(axis='x', rotation=30)

    # Stats table (render as text)
    ax3.axis('off')
    txt = desc.round(3).to_string()
    ax3.text(0, 0.5, txt, fontfamily='monospace')
    ax3.set_title('Summary statistics')

    plt.suptitle(f'Detailed analysis — {name}', y=1.02)
    if save:
        out_dir = Path('notebooks')
        out_dir.mkdir(parents=True, exist_ok=True)
        out = out_dir / f'detail_{name.replace(" ","_")}.png'
        fig.savefig(out, dpi=150, bbox_inches='tight')
        try:
            print('Saved', out.resolve())
        except Exception:
            print('Saved', str(out.resolve()))
    plt.close(fig)

# Chạy cho từng nhóm
for name, cols in groups.items():
    plot_detailed(name, cols, save=True)


Saved C:\Users\Admin\Financial-Distress-Prediction\notebooks\notebooks\detail_ROA.png
Saved C:\Users\Admin\Financial-Distress-Prediction\notebooks\notebooks\detail_Vòng_quay_tài_sản.png
Saved C:\Users\Admin\Financial-Distress-Prediction\notebooks\notebooks\detail_Dòng_tiền_từ_HĐKD.png
Saved C:\Users\Admin\Financial-Distress-Prediction\notebooks\notebooks\detail_Khả_năng_thanh_toán.png


In [33]:
# Áp log-transform cho biến lệch và chuẩn hoá (z-score), vẽ so sánh trước/sau (sửa in đường dẫn)
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')
from pathlib import Path

# Load df
csv = Path('data') / 'dulieuketoan_clean.csv'
if not csv.exists():
    csv = Path('..') / 'data' / 'dulieuketoan_clean.csv'
    if not csv.exists():
        raise FileNotFoundError('Không tìm thấy data/dulieuketoan_clean.csv')

df = pd.read_csv(csv)
df.columns = df.columns.str.strip()

# groups
groups = {
    'ROA': [
        'ROA(C) before interest and depreciation before interest',
        'ROA(A) before interest and % after tax',
        'ROA(B) before interest and depreciation after tax'
    ],
    'Vòng quay tài sản': ['Total Asset Turnover'],
    'Dòng tiền từ HĐKD': ['Cash flow rate'],
    'Khả năng thanh toán': ['Current Ratio', 'Quick Ratio']
}
for k, cols in list(groups.items()):
    groups[k] = [c for c in cols if c in df.columns]

# signed_log1p
def signed_log1p(series):
    return np.sign(series) * np.log1p(np.abs(series))

summary = {}
for name, cols in groups.items():
    if not cols:
        summary[name] = {'status': 'no_columns'}
        continue
    orig = df[cols].apply(pd.to_numeric, errors='coerce')
    trans = orig.copy()
    skew_info = {}
    for col in cols:
        s = orig[col].dropna()
        skew = s.skew() if not s.empty else np.nan
        skew_info[col] = float(skew) if pd.notna(skew) else None
        if pd.notna(skew) and abs(skew) > 1:
            trans[col] = signed_log1p(orig[col])
        else:
            trans[col] = orig[col]
        col_vals = trans[col]
        mean = col_vals.mean(skipna=True)
        std = col_vals.std(skipna=True)
        if pd.isna(std) or std == 0:
            trans[col] = col_vals - mean
        else:
            trans[col] = (col_vals - mean) / std
    out_dir = Path('notebooks')
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / f'transform_compare_{name.replace(" ","_")}.png'
    plt.figure(figsize=(12,5))
    ax1 = plt.subplot(1,2,1)
    sns.boxplot(data=orig, ax=ax1)
    ax1.set_title(f'{name} — Original')
    ax1.tick_params(axis='x', rotation=30)
    ax2 = plt.subplot(1,2,2)
    sns.boxplot(data=trans, ax=ax2)
    ax2.set_title(f'{name} — Transformed (log if skew>1) + z-score')
    ax2.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    try:
        print('Saved', out.resolve())
    except Exception:
        print('Saved', str(out.resolve()))

summary


Saved C:\Users\Admin\Financial-Distress-Prediction\notebooks\notebooks\transform_compare_ROA.png
Saved C:\Users\Admin\Financial-Distress-Prediction\notebooks\notebooks\transform_compare_Vòng_quay_tài_sản.png
Saved C:\Users\Admin\Financial-Distress-Prediction\notebooks\notebooks\transform_compare_Dòng_tiền_từ_HĐKD.png
Saved C:\Users\Admin\Financial-Distress-Prediction\notebooks\notebooks\transform_compare_Khả_năng_thanh_toán.png


{}